# I.P.$^+$ --- the baseline with BaCP's fine-tune

The one ablation the paper cannot ship without.

## What it answers

The headline is that $\Delta$ rises monotonically with sparsity: +0.50, +0.80,
+2.04, +9.83 at 0.95/0.97/0.99/0.999. But the two arms do not share a budget.
BaCP runs 50 pruning epochs **plus a 25-epoch mask-frozen AdamW $10^{-4}$
fine-tune**; I.P.\ runs 50 and stops (confirmed: 171/171 I.P.\ records carry
`epochs_ft=0`, 174/174 BaCP records carry `epochs_ft=25`).

Recovery training buys almost nothing on an intact network and a great deal on
a collapsed one. So an unmatched fine-tune predicts *exactly* the curve the
paper attributes to the objective. As it stands the central result and the
central confound are the same object, and no amount of writing separates them.

This arm gives I.P.\ the identical fine-tune. Report
$\Delta' = \mathrm{BaCP} - \mathrm{I.P.}^+$ **replacing** $\Delta$, not beside it.

## Read the 0.999 column first

If $\Delta'$ at 0.999 holds up, the monotonicity claim is real and the paper
can say both arms share an epoch and fine-tune budget. If it collapses toward
zero, the claim was the fine-tune and the paper must reframe around 0.95--0.99.
Either way it is better learned here than from a reviewer.

## No warm start --- the 50 pruning epochs must re-run

There is no shortcut of loading a pruned checkpoint and fine-tuning it.
`BasePruner.__init__` builds all-ones masks, so entering the fine-tune with
`epochs=0` would silently densify the model and report a dense number. Each
cell runs the full pruning phase again with the fine-tune appended.

## Scope

ResNet-34 and ResNet-50, all three criteria, all four sparsities, three seeds:
72 cells, roughly 6.5 h. 20 of the 24 largest deltas live in these two
backbones. The VGGs are a separate job --- they cost 4$\times$ the ResNets per
cell, and VGG-11 is the negative case where the fine-tune matters least.


In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'nb_common.py').exists():
        sys.path.insert(0, str(cand)); break
    if (cand / 'project' / 'test_notebooks' / 'nb_common.py').exists():
        sys.path.insert(0, str(cand / 'project' / 'test_notebooks')); break
else:
    raise RuntimeError('cannot find nb_common.py -- start the kernel inside the repo')
import nb_common as nb
info = nb.setup()

## Plan

Only the fine-tune is added. Everything else is asserted identical to the I.P.\ arm already in Table 1, so $\Delta'$ isolates one change.

In [ ]:
MODELS   = ('resnet34', 'resnet50')
PRUNERS  = ('magnitude', 'snip', 'wanda')
SPARSITIES = (0.95, 0.97, 0.99, 0.999)
SEEDS    = (1, 2, 3)
GPU      = 0

FT = dict(enable_finetune=True, epochs_ft=25,
          optimizer_type_ft='adamw', learning_rate_ft=1e-4)

plan = []
for m in MODELS:
    for p in PRUNERS:
        for sp in SPARSITIES:
            for s in SEEDS:
                plan.append(nb.make_cell(m, 'prune', seed=s, pruner=p, sparsity=sp,
                                         variant='ft25', **FT))

# the fine-tune must be the ONLY difference from the reported I.P. arm
for c in plan:
    cfg = c['config']
    ref = nb.FAMILIES[c['model_name']]['prune']
    for k in ('learning_rate', 'epochs', 'delta_T', 'sparsity_scheduler',
              'recovery_epochs', 'val_split', 'prune_task_head', 'wanda_group',
              'optimizer_type', 'batch_size', 'num_classes', 'dataset_name'):
        if k in ref:
            assert cfg[k] == ref[k], (c['key'], k, cfg[k], ref[k])
    assert cfg['enable_finetune'] is True
    assert cfg['epochs_ft'] == 25
    assert c['key'].endswith('.ft25'), c['key']

print(f'{len(plan)} cells')
print(f'est ~{sum(2.78 if c["model_name"]=="resnet34" else 4.48 for c in plan)*1.5/60:.1f} h '
      f'(measured I.P.: r34 2.78 min, r50 4.48 min; +50% for 25 fine-tune epochs)')
assert nb.sanity_check(plan), 'sanity check failed'

## Run

Criterion-major within each model, so a partial run still yields whole criteria.

In [ ]:
nb.run_group(plan, gpu=GPU)

## Verdict --- does the monotonic trend survive?

In [ ]:
import json, glob, os, statistics as st
root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok' and '.smoke' not in k:
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

def cell(arm, m, p, sp, suffix=''):
    xs = [acc.get(f'static.{arm}.{m}.cifar10.s{sp}.{p}.seed{s}{suffix}') for s in SEEDS]
    xs = [x for x in xs if x is not None]
    return (st.mean(xs), st.stdev(xs) if len(xs) > 1 else 0.0, len(xs)) if xs else None

print(f'{"cell":34s} {"I.P.":>8} {"I.P.+":>8} {"BaCP":>8} {"D":>7} {"D-prime":>8}')
byspar = {}
for sp in SPARSITIES:
    ds, dps = [], []
    for m in MODELS:
        for p in PRUNERS:
            ip  = cell('prune', m, p, sp)
            ipp = cell('prune', m, p, sp, '.ft25')
            bc  = cell('bacp',  m, p, sp)
            if not (ip and bc):
                continue
            d  = bc[0] - ip[0]
            dp = (bc[0] - ipp[0]) if ipp else None
            ds.append(d)
            if dp is not None:
                dps.append(dp)
            f = lambda v: f'{v[0]:8.2f}' if v else '      --'
            print(f'{m+"/"+p+"/"+str(sp):34s} {f(ip)} {f(ipp)} {f(bc)} '
                  f'{d:+7.2f} ' + (f'{dp:+8.2f}' if dp is not None else '      --'))
    byspar[sp] = (st.mean(ds) if ds else None, st.mean(dps) if dps else None)

print()
print(f'{"sparsity":>9} {"mean D":>9} {"mean D-prime":>13}')
for sp in SPARSITIES:
    d, dp = byspar[sp]
    print(f'{sp:>9} {d:+9.2f} ' + (f'{dp:+13.2f}' if dp is not None else '           --'))

vals = [byspar[sp][1] for sp in SPARSITIES if byspar[sp][1] is not None]
if len(vals) == len(SPARSITIES):
    print()
    if vals == sorted(vals):
        print('D-prime still rises monotonically with sparsity.')
        print('The trend is the OBJECTIVE, not the fine-tune. Claim survives;')
        print("report D-prime in place of D and say both arms share the budget.")
    else:
        print('D-prime is NOT monotone in sparsity.')
        print('The fine-tune carried part of the trend. Reframe around the')
        print('sparsities where D-prime holds, and say so plainly.')